# 1.6 实践：审计并改进一个 SpMV Backend

## 实践任务

1. 画出 `benchmark → backend.prepare → backend.run → correctness/CSV` 调用链。
2. 在课程副本中选择 partition 候选数或 precision 路径作为唯一变量。
3. 用同一矩阵、warmup/repeat 和 CPU reference 运行修改前后版本。
4. 记录 total、阶段时间、footprint、balance ratio 和 relative error。
5. 根据构建依赖与调用点说明结果属于 Host 原型还是 Ascend Device。

不得把变量名中的 `npu/device/hbm/kernel` 当作运行位置证据。

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
import platform, shutil
print("Python:", platform.python_version())
print("CMake:", shutil.which("cmake"))
print("正式路径：Ascend C FP32 RTC；FP16/BF16/persistent 仍为 Host Prototype")


In [ ]:
%%bash
set -e
cd src/ascend_spmv
bash scripts/build.sh
bash scripts/run.sh --matrix U1 --warmup 1 --repeat 3 --csv results/chapter_test.csv
head -2 results/chapter_test.csv

## 评价标准

- 只改变一个变量，并保留正确性检查
- 区分 cold start 与 steady state
- 引用真实类、函数和 CSV 字段
- 对执行位置的判断给出代码/链接证据

参考答案见 `answer/01.06_answer.md`。

## 工程实践提交物与完成标准

章测必须基于 `src/ascend_spmv/`，不得只回答概念题。操作链：构建 benchmark → 追踪接口 → 比较精度表示与误差 → 检查 nnz-aware 边界 → 区分 cold/warm → 分析历史 CSV → 说明执行边界。

提交物：实际命令与环境；阅读或修改的真实文件/函数/参数；字段为“Backend、Precision、Cold、Warm、Total、Compression、Balance、Error”的结果表；正确性判据；基于数据的结论。性能数字不作为固定答案。

完成标准：命令指向真实脚本或可执行文件，数据来自同口径运行，并能解释结果。


## 四类考核

以下四题中，客观题答案唯一，凭 `src/ascend_spmv/` 源码与输出即可判定；简单/中等/困难题基于本章实验，要求用命令、输出、CSV 或计算过程作为证据，不接受无证据的概念回答。

### 1. 客观题

（1）单选：正式运行 `spmv_benchmark` 后，CSV 中 `actual_backend` 字段的值是（　）

A. `host_only`　　B. `ascend_c`　　C. `cpu_openmp16`　　D. `host_prototype_bf16_persistent`

（2）短填空：FP32 正式路径的真实后端类名是 `____`（定义于 `include/ascendc_spmv.hpp`）。

（3）判断（对/错）：`ascendc_launch_to_complete_ms` 记录的是 kernel 从提交到 `aclrtSynchronizeStream` 完成（launch-to-complete）的时间，不是纯 kernel 执行时间。（　）

### 2. 简单题

给出一次运行命令与 CSV 头部及一行数据，逐字段解释 cold/warm、total、compression、balance、error 的含义，并指出哪些字段本身能证明 Device 执行、哪些不能。

### 3. 中等题

选择 partition 候选数或精度路径为唯一变量，同一矩阵与同一 warmup/repeat 运行两次，给出字段级对比（total、balance、error），并用构建/调用点证据说明结果属于哪条实现路径。

### 4. 困难题

审计口径：通读 benchmark 与脚本，列出 cold/warm 与执行位置相关字段的真实语义，指出哪些历史 CSV 不能作为当前 NPU 结果；设计一个可复核的判定（例如 RTC 编译/加载日志证据），并用证据回答“这次运行到底在哪里计算”。
